# SMA-ML/DL — XGBoost + LSTM con Bayesian Optimization (Optuna + GPU)
Ciclo de vida completo:
- **Agente 3 XGBoost**: Optuna TPE, 50 trials, K=5 TimeSeriesSplit
- **Agente 4 LSTM**: Optuna TPE, 30 trials, K=5 folds temporales, GPU T4
- **Ensemble**: scipy optimize pesos en val 2018-2020
- Sube todos los artefactos a AWS S3

In [ ]:
!pip install boto3 python-dotenv scikit-learn optuna xgboost -q

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["AWS_ACCESS_KEY_ID"]     = userdata.get("AWS_ACCESS_KEY_ID")
    os.environ["AWS_SECRET_ACCESS_KEY"] = userdata.get("AWS_SECRET_ACCESS_KEY")
    os.environ["AWS_DEFAULT_REGION"]    = userdata.get("AWS_DEFAULT_REGION")
    os.environ["S3_BUCKET_NAME"]        = userdata.get("S3_BUCKET_NAME")
    print("Credenciales cargadas desde Colab Secrets")
except Exception:
    os.environ["AWS_ACCESS_KEY_ID"]     = "PEGA_AQUI"
    os.environ["AWS_SECRET_ACCESS_KEY"] = "PEGA_AQUI"
    os.environ["AWS_DEFAULT_REGION"]    = "us-east-2"
    os.environ["S3_BUCKET_NAME"]        = "epipredict-dengue"
    print("Credenciales pegadas manualmente")

In [ ]:
import boto3
BUCKET = os.environ["S3_BUCKET_NAME"]
s3 = boto3.client("s3",
    aws_access_key_id=os.environ["AWS_ACCESS_KEY_ID"],
    aws_secret_access_key=os.environ["AWS_SECRET_ACCESS_KEY"],
    region_name=os.environ["AWS_DEFAULT_REGION"])
os.makedirs("/content/modelos", exist_ok=True)
print("Descargando dataset...")
s3.download_file(BUCKET, "datos_procesados/dataset_features_latam.csv",
                 "/content/dataset_features_latam.csv")
print("OK")

In [ ]:
import numpy as np, pandas as pd, pickle, json, warnings
import torch, torch.nn as nn
import xgboost as xgb
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
from scipy.optimize import minimize_scalar
warnings.filterwarnings("ignore")

DEVICE  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEMILLA = 42
SEQ_LEN = 12
LSTM_FEATS = ["tmax_promedio","tmin_promedio","precipitacion",
              "humedad_promedio","agua_basica","incidencia_dengue"]
COLS_EXCLUIR = ["iso_a0","pais","adm_1_name","ano","mes",
                "casos_dengue","poblacion","incidencia_dengue"]
torch.manual_seed(SEMILLA); np.random.seed(SEMILLA)
print(f"Device: {DEVICE}")
if DEVICE.type == "cuda": print(torch.cuda.get_device_name(0))

In [ ]:
df = pd.read_csv("/content/dataset_features_latam.csv")
COLS_FEAT = [c for c in df.columns if c not in COLS_EXCLUIR]
print(f"Features disponibles: {len(COLS_FEAT)}")

df_train = df[df["ano"] <= 2020].copy().reset_index(drop=True)
df_test  = df[df["ano"] >= 2021].copy().reset_index(drop=True)
anos_train_xgb = df_train["ano"].values

X_train_xgb = df_train[COLS_FEAT].values
y_train_xgb = df_train["incidencia_dengue"].values
X_test_xgb  = df_test[COLS_FEAT].values
y_test_xgb  = df_test["incidencia_dengue"].values
y_train_log_xgb = np.log1p(y_train_xgb)

print(f"XGB  Train: {len(X_train_xgb)} | Test: {len(X_test_xgb)}")

## Agente 3 — XGBoost con Bayesian Optimization (50 trials, K=5)

In [ ]:
print("[XGB Fase 1] Baseline XGBoost (params por defecto, 100 arboles)...")
bl_xgb = xgb.XGBRegressor(n_estimators=100, random_state=SEMILLA, verbosity=0)
bl_xgb.fit(X_train_xgb, y_train_log_xgb)
pred_bl_xgb = bl_xgb.predict(X_test_xgb)
r2_bl_xgb  = r2_score(np.log1p(y_test_xgb), pred_bl_xgb)
mae_bl_xgb = mean_absolute_error(y_test_xgb, np.expm1(pred_bl_xgb))
print(f"[XGB Fase 2] Baseline R2={r2_bl_xgb*100:.2f}%  MAE={mae_bl_xgb:.2f}")

In [ ]:
print("[XGB Fase 3] Bayesian Optimization (Optuna/TPE) — 50 trials x K=5...")

folds_xgb = [
    (anos_train_xgb <= 2016, anos_train_xgb == 2016),
    (anos_train_xgb <= 2016, anos_train_xgb == 2017),
    (anos_train_xgb <= 2017, anos_train_xgb == 2018),
    (anos_train_xgb <= 2018, anos_train_xgb == 2019),
    (anos_train_xgb <= 2019, anos_train_xgb == 2020),
]

def objective_xgb(trial):
    params = {
        "n_estimators":      trial.suggest_int("n_estimators", 200, 1200),
        "learning_rate":     trial.suggest_float("learning_rate", 0.001, 0.1, log=True),
        "max_depth":         trial.suggest_int("max_depth", 3, 8),
        "min_child_weight":  trial.suggest_int("min_child_weight", 1, 10),
        "subsample":         trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree":  trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "gamma":             trial.suggest_float("gamma", 0.0, 0.5),
        "objective": "reg:squarederror",
        "random_state": SEMILLA,
        "verbosity": 0,
        "n_jobs": -1,
    }
    r2s = []
    for tr_m, vl_m in folds_xgb:
        m = xgb.XGBRegressor(**params)
        m.fit(X_train_xgb[tr_m], y_train_log_xgb[tr_m])
        pv = m.predict(X_train_xgb[vl_m])
        r2s.append(r2_score(y_train_log_xgb[vl_m], pv))
    return float(np.mean(r2s))

study_xgb = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=SEMILLA)
)

def cb_xgb(study, trial):
    best = " <-- mejor" if trial.value == study.best_value else ""
    p = trial.params
    print(f"  Trial {trial.number+1:02d}  "
          f"n_est={p['n_estimators']}  lr={p['learning_rate']:.4f}  "
          f"depth={p['max_depth']}  R2_CV={trial.value*100:.2f}%{best}")

study_xgb.optimize(objective_xgb, n_trials=50, callbacks=[cb_xgb])
best_xgb = study_xgb.best_trial
print(f"\nMejor trial XGB: R2_CV={best_xgb.value*100:.2f}%")
print(f"Params: {best_xgb.params}")

In [ ]:
print("[XGB Fase 4] Reentrenando XGBoost con mejores hiperparametros...")
best_params_xgb = {
    **best_xgb.params,
    "objective": "reg:squarederror",
    "random_state": SEMILLA,
    "verbosity": 0,
    "n_jobs": -1,
}
model_xgb = xgb.XGBRegressor(**best_params_xgb)

# Pipeline con StandardScaler (igual que el sistema en produccion)
scaler_xgb = StandardScaler()
X_tr_xgb_sc = scaler_xgb.fit_transform(X_train_xgb)
X_te_xgb_sc  = scaler_xgb.transform(X_test_xgb)
model_xgb.fit(X_tr_xgb_sc, y_train_log_xgb)

pred_xgb_log = model_xgb.predict(X_te_xgb_sc)
pred_xgb_real = np.maximum(np.expm1(pred_xgb_log), 0)
r2_xgb  = r2_score(np.log1p(y_test_xgb), pred_xgb_log)
mae_xgb = mean_absolute_error(y_test_xgb, pred_xgb_real)
rmse_xgb = np.sqrt(mean_squared_error(y_test_xgb, pred_xgb_real))
print(f"XGBoost Bayesian  R2={r2_xgb*100:.2f}%  MAE={mae_xgb:.2f}  RMSE={rmse_xgb:.2f}")

# Guardar pipeline
from sklearn.pipeline import Pipeline
pipe_xgb = Pipeline([("scaler", scaler_xgb), ("model", model_xgb)])
with open("/content/modelos/pipeline_ml.pkl","wb") as f: pickle.dump(pipe_xgb,f)
print("Pipeline XGBoost guardado")

## Agente 4 — LSTM con Bayesian Optimization (30 trials, K=5, GPU)

In [ ]:
class DengueLSTMModel(nn.Module):
    def __init__(self, input_dim=6, hidden_dim=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers,
                            batch_first=True,
                            dropout=dropout if num_layers > 1 else 0.0)
        self.fc = nn.Linear(hidden_dim, 1)
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :]).squeeze(-1)

def build_sequences(df, features, seq_len):
    X_list, y_list, ano_list = [], [], []
    for (iso, adm), grp in df.groupby(["iso_a0","adm_1_name"]):
        grp = grp.sort_values(["ano","mes"]).reset_index(drop=True)
        vals = grp[features].values
        inc  = grp["incidencia_dengue"].values
        anos = grp["ano"].values
        for i in range(seq_len, len(grp)):
            X_list.append(vals[i-seq_len:i])
            y_list.append(inc[i])
            ano_list.append(anos[i])
    return (np.array(X_list, dtype=np.float32),
            np.array(y_list, dtype=np.float32),
            np.array(ano_list))

def fit_lstm(X, y_log, hidden, lr, dropout, num_layers, epochs, device):
    torch.manual_seed(SEMILLA)
    m = DengueLSTMModel(6, hidden, num_layers=num_layers, dropout=dropout).to(device)
    opt = torch.optim.Adam(m.parameters(), lr=lr, weight_decay=1e-4)
    sch = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, patience=10, factor=0.5)
    Xt = torch.tensor(X, dtype=torch.float32).to(device)
    yt = torch.tensor(y_log, dtype=torch.float32).to(device)
    best_loss, pat, best_sd = np.inf, 0, None
    for ep in range(epochs):
        m.train(); opt.zero_grad()
        loss = nn.MSELoss()(m(Xt), yt)
        loss.backward(); opt.step(); sch.step(loss)
        if loss.item() < best_loss - 1e-5:
            best_loss = loss.item()
            best_sd = {k: v.clone() for k, v in m.state_dict().items()}
            pat = 0
        else:
            pat += 1
        if pat >= 15: break
    if best_sd: m.load_state_dict(best_sd)
    return m

print("Clases LSTM OK")

In [ ]:
print("Construyendo secuencias LSTM...")
X_seq, y_seq, anos_seq = build_sequences(df, LSTM_FEATS, SEQ_LEN)
X_train_lstm = X_seq[anos_seq <= 2020]
y_train_lstm = y_seq[anos_seq <= 2020]
anos_train_lstm = anos_seq[anos_seq <= 2020]
X_test_lstm  = X_seq[anos_seq >= 2021]
y_test_lstm  = y_seq[anos_seq >= 2021]
escalador_lstm = StandardScaler()
X_tr_lstm_sc = escalador_lstm.fit_transform(
    X_train_lstm.reshape(-1,6)).reshape(X_train_lstm.shape)
X_te_lstm_sc = escalador_lstm.transform(
    X_test_lstm.reshape(-1,6)).reshape(X_test_lstm.shape)
y_tr_lstm_log = np.log1p(y_train_lstm)
print(f"LSTM  Train: {len(X_train_lstm)} | Test: {len(X_test_lstm)}")

In [ ]:
print("[LSTM Fase 1] Baseline (hidden=32, 40 epocas)...")
bl_lstm = fit_lstm(X_tr_lstm_sc, y_tr_lstm_log, 32, 0.01, 0.0, 1, 40, DEVICE)
bl_lstm.eval()
with torch.no_grad():
    p_bl_lstm = bl_lstm(
        torch.tensor(X_te_lstm_sc, dtype=torch.float32).to(DEVICE)).cpu().numpy()
r2_bl_lstm  = r2_score(np.log1p(y_test_lstm), p_bl_lstm)
mae_bl_lstm = mean_absolute_error(y_test_lstm, np.expm1(p_bl_lstm))
print(f"[LSTM Fase 2] Baseline R2={r2_bl_lstm*100:.2f}%  MAE={mae_bl_lstm:.2f}")

In [ ]:
print("[LSTM Fase 3] Bayesian Optimization (Optuna/TPE) — 30 trials x K=5...")

folds_lstm = [
    (anos_train_lstm <= 2016, anos_train_lstm == 2016),
    (anos_train_lstm <= 2016, anos_train_lstm == 2017),
    (anos_train_lstm <= 2017, anos_train_lstm == 2018),
    (anos_train_lstm <= 2018, anos_train_lstm == 2019),
    (anos_train_lstm <= 2019, anos_train_lstm == 2020),
]

def objective_lstm(trial):
    hidden     = trial.suggest_int("hidden_dim", 64, 512, log=True)
    num_layers = trial.suggest_int("num_layers", 1, 3)
    lr         = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
    dropout    = trial.suggest_float("dropout", 0.0, 0.4)
    r2s = []
    for tr_m, vl_m in folds_lstm:
        sc2 = StandardScaler()
        Xtr = sc2.fit_transform(
            X_tr_lstm_sc[tr_m].reshape(-1,6)).reshape(X_tr_lstm_sc[tr_m].shape)
        Xvl = sc2.transform(
            X_tr_lstm_sc[vl_m].reshape(-1,6)).reshape(X_tr_lstm_sc[vl_m].shape)
        m = fit_lstm(Xtr, y_tr_lstm_log[tr_m],
                     hidden, lr, dropout, num_layers, 100, DEVICE)
        m.eval()
        with torch.no_grad():
            pv = m(torch.tensor(Xvl,dtype=torch.float32).to(DEVICE)).cpu().numpy()
        r2s.append(r2_score(y_tr_lstm_log[vl_m], pv))
    return float(np.mean(r2s))

study_lstm = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=SEMILLA)
)

def cb_lstm(study, trial):
    best = " <-- mejor" if trial.value == study.best_value else ""
    p = trial.params
    print(f"  Trial {trial.number+1:02d}  "
          f"hidden={p['hidden_dim']}  layers={p['num_layers']}  "
          f"lr={p['lr']:.5f}  dropout={p['dropout']:.2f}  "
          f"R2_CV={trial.value*100:.2f}%{best}")

study_lstm.optimize(objective_lstm, n_trials=30, callbacks=[cb_lstm])
best_lstm = study_lstm.best_trial
print(f"\nMejor trial LSTM: R2_CV={best_lstm.value*100:.2f}%")
print(f"Params: {best_lstm.params}")

In [ ]:
print("[LSTM Fase 4] Reentrenando con mejores hiperparametros (300 epocas)...")
bp = best_lstm.params
model_lstm = fit_lstm(X_tr_lstm_sc, y_tr_lstm_log,
                      bp["hidden_dim"], bp["lr"], bp["dropout"],
                      bp["num_layers"], 300, DEVICE)
model_lstm.eval()
with torch.no_grad():
    pred_lstm_log = model_lstm(
        torch.tensor(X_te_lstm_sc,dtype=torch.float32).to(DEVICE)).cpu().numpy()
pred_lstm_real = np.maximum(np.expm1(pred_lstm_log), 0)
r2_lstm   = r2_score(np.log1p(y_test_lstm), pred_lstm_log)
mae_lstm  = mean_absolute_error(y_test_lstm, pred_lstm_real)
rmse_lstm = np.sqrt(mean_squared_error(y_test_lstm, pred_lstm_real))
print(f"LSTM Bayesian  R2={r2_lstm*100:.2f}%  MAE={mae_lstm:.2f}  RMSE={rmse_lstm:.2f}")

## Ensemble — Optimizar pesos XGBoost + LSTM (scipy, val 2018-2020)

In [ ]:
print("[Ensemble] Alineando predicciones y optimizando pesos...")

# Predicciones XGBoost alineadas al test set LSTM
# (LSTM tiene seq_len=12 offset, necesitamos alinear por ano/mes/departamento)
df_te_full = df[df["ano"] >= 2021].copy()
xgb_log_te = pipe_xgb.predict(df_te_full[COLS_FEAT])
xgb_lkp = {
    (str(r.iso_a0).upper(), str(r.adm_1_name).upper(), int(r.ano), int(r.mes)): float(p)
    for r, p in zip(df_te_full.itertuples(), xgb_log_te)
}

# Reconstruir seq_ids para alineacion
seq_ids_te = []
for (iso,adm), grp in df.groupby(["iso_a0","adm_1_name"]):
    grp = grp.sort_values(["ano","mes"]).reset_index(drop=True)
    anos_g = grp["ano"].values
    meses_g = grp["mes"].values
    for i in range(SEQ_LEN, len(grp)):
        if anos_g[i] >= 2021:
            seq_ids_te.append((str(iso).upper(),str(adm).upper(),int(anos_g[i]),int(meses_g[i])))

y_alin, xgb_alin, lstm_alin = [], [], []
for sid, lp in zip(seq_ids_te, pred_lstm_log):
    xp = xgb_lkp.get(sid)
    if xp is None: continue
    iso,adm,ano,mes = sid
    row = df_te_full[
        (df_te_full["iso_a0"].str.upper()==iso) &
        (df_te_full["adm_1_name"].str.upper()==adm) &
        (df_te_full["ano"]==ano) & (df_te_full["mes"]==mes)]
    if len(row):
        y_alin.append(float(row["incidencia_dengue"].iloc[0]))
        xgb_alin.append(xp); lstm_alin.append(lp)

y_alin = np.array(y_alin)
ly_alin = np.log1p(y_alin)
xgb_alin = np.array(xgb_alin)
lstm_alin = np.array(lstm_alin)
print(f"Registros alineados: {len(y_alin)}")

# Validation set 2018-2020 para optimizar pesos
df_val = df[df["ano"].isin([2018,2019,2020])].copy()
xgb_log_v = pipe_xgb.predict(df_val[COLS_FEAT])
xgb_lkp_v = {
    (str(r.iso_a0).upper(), str(r.adm_1_name).upper(), int(r.ano), int(r.mes)): float(p)
    for r, p in zip(df_val.itertuples(), xgb_log_v)
}
seq_ids_v, lstm_preds_v = [], []
for (iso,adm), grp in df[df["ano"]<=2020].groupby(["iso_a0","adm_1_name"]):
    grp = grp.sort_values(["ano","mes"]).reset_index(drop=True)
    vals_g = grp[LSTM_FEATS].values
    anos_g = grp["ano"].values
    meses_g = grp["mes"].values
    for i in range(SEQ_LEN, len(grp)):
        if anos_g[i] in [2018,2019,2020]:
            seq_ids_v.append((str(iso).upper(),str(adm).upper(),int(anos_g[i]),int(meses_g[i])))

X_v_seq = np.array([grp[LSTM_FEATS].values[i-SEQ_LEN:i]
    for (iso,adm), grp in df[df["ano"]<=2020].groupby(["iso_a0","adm_1_name"])
    for i in range(SEQ_LEN, len(grp.sort_values(["ano","mes"])))
    if grp.sort_values(["ano","mes"])["ano"].values[i] in [2018,2019,2020]],
    dtype=np.float32) if False else None

# Forma mas simple: usar los datos de val directamente
X_sq_all, y_sq_all, a_sq_all = build_sequences(df[df["ano"]<=2020], LSTM_FEATS, SEQ_LEN)
mask_v = (a_sq_all >= 2018)
Xv_sc = escalador_lstm.transform(
    X_sq_all[mask_v].reshape(-1,6)).reshape(X_sq_all[mask_v].shape)
model_lstm.eval()
with torch.no_grad():
    lstm_v = model_lstm(
        torch.tensor(Xv_sc,dtype=torch.float32).to(DEVICE)).cpu().numpy()
ly_v_sq = np.log1p(y_sq_all[mask_v])
nv = min(len(xgb_log_v), len(lstm_v))

def neg_r2_val(w): return -r2_score(ly_v_sq[:nv], w*xgb_log_v[:nv]+(1-w)*lstm_v[:nv])
res = minimize_scalar(neg_r2_val, bounds=(0.4,0.95), method="bounded")
w_xgb_opt, w_lstm_opt = float(res.x), 1.0 - float(res.x)

# Metricas ensemble en test
ens_log  = w_xgb_opt*xgb_alin + w_lstm_opt*lstm_alin
ens_real = np.maximum(np.expm1(ens_log), 0)
r2_ens   = r2_score(ly_alin, ens_log)
mae_ens  = mean_absolute_error(y_alin, ens_real)
rmse_ens = np.sqrt(mean_squared_error(y_alin, ens_real))

print(f"\n{60*"="}")
print(f"  RESULTADOS FINALES (Bayesian Optimization)")
print(f"{60*"="}")
print(f"  XGBoost  R2={r2_xgb*100:.2f}%  MAE={mae_xgb:.2f}  RMSE={rmse_xgb:.2f}")
print(f"  LSTM     R2={r2_lstm*100:.2f}%  MAE={mae_lstm:.2f}  RMSE={rmse_lstm:.2f}")
print(f"  Ensemble R2={r2_ens*100:.2f}%  MAE={mae_ens:.2f}  RMSE={rmse_ens:.2f}")
print(f"  w_xgb={w_xgb_opt:.4f}  w_lstm={w_lstm_opt:.4f}")
print(f"{60*"="}")

In [ ]:
print("[Fase Final] Guardando artefactos y subiendo a S3...")

# XGBoost
with open("/content/modelos/pipeline_ml.pkl","wb") as f: pickle.dump(pipe_xgb,f)

# LSTM
model_lstm.cpu()
torch.save(model_lstm.state_dict(), "/content/modelos/lstm_model.pth")
with open("/content/modelos/escalador_lstm.pkl","wb") as f: pickle.dump(escalador_lstm,f)
with open("/content/modelos/lstm_features.pkl","wb") as f: pickle.dump(LSTM_FEATS,f)

lstm_cfg = {
    "seq_len": SEQ_LEN, "input_dim": 6,
    "hidden_dim": bp["hidden_dim"], "num_layers": bp["num_layers"],
    "lr": bp["lr"], "dropout": bp["dropout"],
    "r2": round(float(r2_lstm),4), "mae": round(float(mae_lstm),4),
    "rmse": round(float(rmse_lstm),4),
    "r2_baseline": round(float(r2_bl_lstm),4),
    "r2_cv_mejor": round(float(best_lstm.value),4),
    "k_folds": 5, "optimizer": "Optuna/TPE", "n_trials": 30,
    "trials": [{"trial":t.number,"params":t.params,"r2_cv":t.value}
               for t in study_lstm.trials]
}
with open("/content/modelos/lstm_config.json","w") as f: json.dump(lstm_cfg,f,indent=2)

xgb_cfg = {
    "best_params": best_xgb.params,
    "r2": round(float(r2_xgb),4), "mae": round(float(mae_xgb),4),
    "rmse": round(float(rmse_xgb),4),
    "r2_baseline": round(float(r2_bl_xgb),4),
    "r2_cv_mejor": round(float(best_xgb.value),4),
    "k_folds": 5, "optimizer": "Optuna/TPE", "n_trials": 50,
    "trials": [{"trial":t.number,"params":t.params,"r2_cv":t.value}
               for t in study_xgb.trials]
}
with open("/content/modelos/xgb_config.json","w") as f: json.dump(xgb_cfg,f,indent=2)

# metrics.json
metrics = {
    "records_procesados": len(df),
    "r2_xgb":         round(float(r2_xgb),4),
    "mae_xgb":        round(float(mae_xgb),4),
    "rmse_xgb":       round(float(rmse_xgb),4),
    "r2_lstm":        round(float(r2_lstm),4),
    "mae_lstm":       round(float(mae_lstm),4),
    "rmse_lstm":      round(float(rmse_lstm),4),
    "r2_ensemble":    round(float(r2_ens),4),
    "mae_ensemble":   round(float(mae_ens),4),
    "rmse_ensemble":  round(float(rmse_ens),4),
    "ensemble_w_xgb": round(w_xgb_opt,4),
    "ensemble_w_lstm":round(w_lstm_opt,4),
    "n_train": int((df["ano"]<=2020).sum()),
    "n_test":  int((df["ano"]>=2021).sum()),
    "n_paises": int(df["pais"].nunique()),
    "n_departamentos": int(df["adm_1_name"].nunique())
}
with open("/content/modelos/metrics.json","w") as f: json.dump(metrics,f,indent=2)

# Subir a S3
archivos = [
    "pipeline_ml.pkl","lstm_model.pth","escalador_lstm.pkl",
    "lstm_features.pkl","lstm_config.json","xgb_config.json","metrics.json"
]
for fname in archivos:
    s3.upload_file(f"/content/modelos/{fname}", BUCKET, f"modelos/{fname}")
    print(f"  Subido: {fname}")

print(f"\nDONE")
print(f"  XGBoost  R2={r2_xgb*100:.2f}%")
print(f"  LSTM     R2={r2_lstm*100:.2f}%")
print(f"  Ensemble R2={r2_ens*100:.2f}%  (w_xgb={w_xgb_opt:.2f})")